# gpt-oss-20b on vLLM, for the Journey Console

Run the cells in order (Shift+Enter). They:

1. start **gpt-oss-20b** (`openai/gpt-oss-20b`) in a **vLLM server** on this machine (an OpenAI-compatible endpoint,
   `http://localhost:8000/v1`), in its own process, so it keeps running for the console and its test runs;
2. check it answers the way the test tools need (a JSON verdict, through the same client the console uses);
3. make it the console's model for **rewording** (the LLM arm: rephrase, dual read, adversary) and/or the
   **judges** (the two LLM judges, the policy judge, the case summaries): the same setting as the pickers
   *Rewording LLM* and *Judges' LLM* on the run page, kept in `output/console_llm.json`;
4. optionally make it the **bot under test** (its card on the run page, and the Playground);
5. open the console; stop the server when you are done.

**Needs:** an NVIDIA GPU with about 24 GB free (the 20B model is about 13 GB in MXFP4, plus its KV cache),
**vLLM 0.10.1 or newer** (gpt-oss support), and the weights: downloaded from Hugging Face on the first start (about
13 GB), or a local copy (set `MODEL` to its folder).

gpt-oss is a *reasoning* model: it thinks before it answers. The console's client asks it for
`reasoning_effort: "low"` and gives it a thinking budget of 1024 tokens on top of each answer
(`GMSJOURNEY_VLLM_REASONING_EFFORT`, `GMSJOURNEY_VLLM_REASONING_TOKENS`), so a short judge verdict is not cut off by
its own reasoning.

In [ ]:
# 1. SETTINGS
import os, sys
if os.path.exists("../gmsjourney/__init__.py") and not os.path.exists("gmsjourney/__init__.py"):
    os.chdir("..")                                     # Jupyter started in notebooks/: work from the repository root
sys.path.insert(0, os.path.abspath("."))

MODEL = "openai/gpt-oss-20b"        # the Hugging Face id, or the folder of a local copy of the weights
SERVED_NAME = "openai/gpt-oss-20b"  # the name the server answers to (the console sends it as the model name)
PORT = 8000                         # the server: http://localhost:8000/v1
GPU = "0"                           # which GPU(s): "0", or "0,1" with TENSOR_PARALLEL = 2
TENSOR_PARALLEL = 1
GPU_MEMORY_UTILIZATION = 0.85       # share of the GPU's memory vLLM may take
MAX_MODEL_LEN = 32768               # context length (the LLM bot's prompt is about 6k tokens, a judge's 2-4k)
INSTALL_VLLM = False                # True: pip install "vllm>=0.10.1" into this kernel first

USE_FOR_REWORDING = True            # the LLM arm (rephrase, dual read, adversary)
USE_FOR_JUDGES = True               # the two LLM judges, the policy judge, the case summaries
USE_AS_BOT_DEFAULT = False          # also prefill the bot-under-test card's local model with this server
CONSOLE_PORT = 8790

URL = f"http://localhost:{PORT}/v1"
LOG = os.path.abspath("output/vllm_gpt_oss.log")
PID = os.path.abspath("output/vllm_gpt_oss.pid")
os.makedirs("output", exist_ok=True)
print("server:", URL, "| log:", LOG)

In [ ]:
# 2. CHECK THE MACHINE: a GPU, and vLLM new enough for gpt-oss
import subprocess, importlib.util
r = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used", "--format=csv,noheader"],
                   capture_output=True, text=True)
print(r.stdout.strip() or "no NVIDIA GPU found (nvidia-smi): vLLM needs one for gpt-oss-20b")
if INSTALL_VLLM:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm>=0.10.1"], check=True)
if importlib.util.find_spec("vllm") is None:
    print("vLLM is not installed in this kernel: set INSTALL_VLLM = True above (or pip install 'vllm>=0.10.1')")
else:
    import vllm
    v = tuple(int(x) for x in vllm.__version__.split("+")[0].split(".")[:3] if x.isdigit())
    print("vLLM", vllm.__version__, "- ok" if v >= (0, 10, 1) else "- too old for gpt-oss: install vllm>=0.10.1")

In [ ]:
# 3. START THE SERVER (in its own process; running this cell again reuses a server that is already up)
import json, time, signal, urllib.request

def up(url=URL):
    try:
        with urllib.request.urlopen(url + "/models", timeout=3) as r:
            return [m["id"] for m in json.loads(r.read())["data"]]
    except Exception:
        return None

def tail(n=15):
    try:
        return "".join(open(LOG, encoding="utf-8", errors="replace").readlines()[-n:])
    except OSError:
        return ""

if up():
    print("already serving:", up())
else:
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": GPU}
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL,
           "--served-model-name", SERVED_NAME, "--port", str(PORT), "--host", "127.0.0.1",
           "--tensor-parallel-size", str(TENSOR_PARALLEL), "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
           "--max-model-len", str(MAX_MODEL_LEN)]
    proc = subprocess.Popen(cmd, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, env=env, start_new_session=True)
    open(PID, "w").write(str(proc.pid))
    print("starting:", " ".join(cmd))
    t0, last = time.time(), 0
    while not up():
        if proc.poll() is not None:
            raise RuntimeError("the vLLM server stopped:\n" + tail(40))
        if time.time() - t0 > 30 * 60:
            raise TimeoutError("not up after 30 minutes:\n" + tail(40))
        if time.time() - last > 60:                    # the first start downloads the weights: show progress
            last = time.time()
            print(f"... {int(time.time() - t0)} s\n" + tail(3))
        time.sleep(5)
    print(f"up after {int(time.time() - t0)} s, serving:", up())

In [ ]:
# 4. CHECK IT ANSWERS THE WAY THE TEST TOOLS NEED (the console's own client, so this is what a run will see)
from gmsjourney.s06_bots.llm_bot import LocalChat
from gmsjourney.llm import _messages, parse_json_object
chat = LocalChat(SERVED_NAME, temperature=0.0, max_tokens=300, url=URL)
t0 = time.time()
reply = chat.invoke(_messages(
    'You judge a bot\'s reply. Reply with ONE JSON object: {"verdict": "correct" or "incorrect", "reason": "..."}',
    "Customer: I don't recognise the Amazon charge.\nBot: I've recorded that you don't recognise it."))
print(f"{time.time() - t0:.1f} s, {reply.usage_metadata}")
print(reply.content)
print("parsed:", parse_json_object(reply.content) or "NOT JSON: raise GMSJOURNEY_VLLM_REASONING_TOKENS or check the log")

In [ ]:
# 5. MAKE IT THE CONSOLE'S MODEL (rewording and/or judges), now and for the next console start
from gmsjourney.s10_console import server as console_server
choice = {"backend": "local", "local_path": SERVED_NAME, "vllm_url": URL, "model": ""}

def console_url():
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{CONSOLE_PORT}/api/meta", timeout=2)
        return f"http://127.0.0.1:{CONSOLE_PORT}/"
    except Exception:
        return None

for role, on in (("enrichment", USE_FOR_REWORDING), ("testing", USE_FOR_JUDGES)):
    if not on:
        continue
    body = {**choice, "role": role}
    base = console_url()
    if base:                                           # a console is running: tell it (it saves the choice)
        req = urllib.request.Request(base + "api/llm/settings", data=json.dumps(body).encode(),
                                     headers={"Content-Type": "application/json"})
        now = json.loads(urllib.request.urlopen(req).read())[role]
    else:                                              # not running: save it for the console's next start
        now = console_server.save_tools_llm(body)[role]
    print(f"{'rewording' if role == 'enrichment' else 'judges'}: {now['backend']} {now.get('local_path')} "
          f"at {now.get('vllm_url')} - {'ready' if now.get('usable') else now.get('detail')}")

if USE_AS_BOT_DEFAULT:                                 # the bot card's local model starts filled in with this server
    os.environ["GMSJOURNEY_SUT_LOCAL_MODEL"] = SERVED_NAME
    os.environ["GMSJOURNEY_VLLM_URL"] = URL
print("\nAs the bot under test: run page -> Bot under test -> LLM bot -> Local model, "
      f"model folder or name '{SERVED_NAME}', vLLM server URL '{URL}'.")

## Open the console

The run page shows the choice under **LLM rewording of customer turns** (*Rewording LLM*) and **Two LLM judges**
(*Judges' LLM*); the Playground's side panel shows the judges' model. Switch either back to the Microsoft gateway
there at any time. A console that was already open picks the choice up immediately (cell 5 told it); reload its tab
to see it.

In [ ]:
# 6. OPEN THE CONSOLE (it inherits this kernel's settings)
from gmsjourney import s10_console as console
console.show(port=CONSOLE_PORT, height=960)

In [ ]:
# 7. STOP THE SERVER (and put the console's rewording and judges back on the Microsoft gateway)
RESET_TO_GATEWAY = True
try:
    os.killpg(int(open(PID).read()), signal.SIGTERM)
    print("vLLM server stopped")
except (OSError, ValueError) as e:
    print("no server to stop:", e)
if RESET_TO_GATEWAY:
    for role in ("enrichment", "testing"):
        body = {"role": role, "backend": "gateway"}
        base = console_url()
        if base:
            urllib.request.urlopen(urllib.request.Request(base + "api/llm/settings", data=json.dumps(body).encode(),
                                                          headers={"Content-Type": "application/json"}))
        else:
            console_server.save_tools_llm(body)
    print("rewording and judges: Microsoft gateway")